# Fetch ChEMBL activities

IC50 and Ki values for human p38α (CHEMBL260), COX-2 (CHEMBL230) and 5-LOX (CHEMBL215) from
the ChEMBL API, collapsed to one row per molecule. Each target writes
`data/{slug}_training_raw.csv`, with slug `p38`, `cox2` or `lox5`.

In [ ]:
TARGETS = {
    'p38':  {'chembl_id': 'CHEMBL260', 'name': 'p38alpha (MAPK14)'},
    'cox2': {'chembl_id': 'CHEMBL230', 'name': 'COX-2 (PTGS2)'},
    'lox5': {'chembl_id': 'CHEMBL215', 'name': '5-LOX (ALOX5)'},
}

import os
if os.environ.get('NL_SLUGS'):
    TARGETS = {k: v for k, v in TARGETS.items() if k in os.environ['NL_SLUGS'].split(',')}

REFRESH = False
REFRESH = os.environ.get('NL_REFRESH', '1' if REFRESH else '0') == '1'

for slug, t in TARGETS.items():
    print(f'{slug:5s} {t["chembl_id"]:10s} {t["name"]}')
print(f'REFRESH={REFRESH}')

## Inclusion criteria

- IC50 or Ki, in nM, relation `=`
- human target
- assay confidence score of at least 8
- no data validity comment, not a potential duplicate
- positive value

pActivity = 9 − log10(value in nM). Active is pActivity ≥ 6, inactive is below 5, and the 5–6
band is labelled `dropped` and kept in the file.

In [ ]:
import datetime as dt
import json
import math
import pathlib
import time

import pandas as pd
import requests
from IPython.display import Markdown, display

BASE = 'https://www.ebi.ac.uk/chembl/api/data'
PAGE = 1000
CONFIDENCE_MIN = 8
ACTIVE_CUTOFF = 6.0
INACTIVE_CUTOFF = 5.0

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'src' else pathlib.Path.cwd()
PROVENANCE = ROOT / 'docs/provenance.md'


def raw_json(slug):
    return ROOT / f'data/raw/{slug}_activities_raw.json'


def out_csv(slug):
    return ROOT / f'data/{slug}_training_raw.csv'


def banner(slug):
    t = TARGETS[slug]
    display(Markdown(f'#### {t["name"]} &middot; `{t["chembl_id"]}` &middot; slug `{slug}`'))


SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'NeuroLead-EthnoHack2026/1.0'})
print('root', ROOT)
for slug in TARGETS:
    print(f'  {slug:5s} reads/writes {out_csv(slug).relative_to(ROOT).as_posix()} '
          f'and {raw_json(slug).relative_to(ROOT).as_posix()}')

## API helpers

In [ ]:
def get(path, params):
    url = f'{BASE}/{path}.json'
    for attempt in range(4):
        r = SESSION.get(url, params=params, timeout=120)
        if r.status_code == 200:
            return r.json()
        if attempt == 3:
            r.raise_for_status()
        time.sleep(2 * (attempt + 1))
    raise RuntimeError('unreachable')


def chembl_release():
    try:
        d = get('status', {})
        return f"{d.get('chembl_db_version', '?')} (release id {d.get('chembl_release_id', '?')})"
    except Exception as exc:
        return f'unknown ({exc})'


def fetch_activities(target):
    out, offset = [], 0
    while True:
        d = get('activity', {'target_chembl_id': target, 'standard_type__in': 'IC50,Ki',
                             'limit': PAGE, 'offset': offset})
        batch = d['activities']
        out.extend(batch)
        total = d['page_meta']['total_count']
        print(f'  activities {len(out)}/{total}', flush=True)
        offset += PAGE
        if offset >= total or not batch:
            return out


def fetch_assay_confidence(assay_ids):
    conf, chunk = {}, 50
    for i in range(0, len(assay_ids), chunk):
        ids = assay_ids[i:i + chunk]
        d = get('assay', {'assay_chembl_id__in': ','.join(ids), 'limit': chunk})
        for a in d['assays']:
            conf[a['assay_chembl_id']] = a.get('confidence_score')
    print(f'  looked up {len(conf)} of {len(assay_ids)} assays', flush=True)
    return conf


def p_activity(value_nm):
    return 9.0 - math.log10(value_nm)


def label(p):
    if p >= ACTIVE_CUTOFF:
        return 'active'
    if p < INACTIVE_CUTOFF:
        return 'inactive'
    return 'dropped'

## Download

In [ ]:
retrieved = dt.date.today().isoformat()
state = {slug: {} for slug in TARGETS}

for slug, t in TARGETS.items():
    banner(slug)
    if REFRESH:
        release = chembl_release()
        print('ChEMBL release:', release)
        acts = fetch_activities(t['chembl_id'])
        raw_json(slug).parent.mkdir(parents=True, exist_ok=True)
        raw_json(slug).write_text(json.dumps(acts), encoding='utf-8')
        print(f'raw payload -> {raw_json(slug)}  ({len(acts)} records)')
    else:
        acts = json.loads(raw_json(slug).read_text(encoding='utf-8'))
        release = 'from saved payload'
        print(f'REFRESH is False: loaded the saved payload, {len(acts):,} records, no API call made')
    state[slug].update(acts=acts, release=release)

## Filtering

In [ ]:
def apply_filters(acts):
    df = pd.DataFrame(acts)
    steps = [('fetched (IC50/Ki, any quality)', len(df))]

    df = df[df['standard_units'] == 'nM']
    steps.append(('units == nM', len(df)))

    df = df[df['standard_relation'] == '=']
    steps.append(("relation == '='", len(df)))

    df = df[df['target_organism'] == 'Homo sapiens']
    steps.append(('organism == Homo sapiens', len(df)))

    df = df[df['data_validity_comment'].isna()]
    steps.append(('no data_validity_comment', len(df)))

    df = df[df['potential_duplicate'] == 0]
    steps.append(('not potential_duplicate', len(df)))

    df = df.copy()
    df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
    df = df[df['standard_value'] > 0]
    df = df[df['canonical_smiles'].notna()]
    steps.append(('positive value + SMILES present', len(df)))
    return df, steps


for slug, s in state.items():
    banner(slug)
    s['df'], s['steps'] = apply_filters(s['acts'])
    for name, n in s['steps']:
        print(f'  {n:>7}  {name}')

## Assay confidence

The confidence score is stored on the assay, so it is joined in from the `/assay` endpoint.

In [ ]:
for slug, s in state.items():
    banner(slug)
    df = s['df']
    assay_ids = sorted(df['assay_chembl_id'].dropna().unique().tolist())
    print(f'{len(assay_ids)} distinct assays to look up')
    conf = fetch_assay_confidence(assay_ids)

    df = df.assign(confidence_score=df['assay_chembl_id'].map(conf))
    df = df[df['confidence_score'] >= CONFIDENCE_MIN]
    s['steps'].append((f'confidence_score >= {CONFIDENCE_MIN}', len(df)))
    s['surviving_activities'] = len(df)
    s['df'] = df
    print(f'{len(df):,} activities survive all criteria')

## One row per molecule

Repeated measurements are collapsed to the median pActivity. Minimum, maximum and spread are
kept alongside.

In [ ]:
COLUMNS = ['molecule_chembl_id', 'molecule_pref_name', 'canonical_smiles', 'target_chembl_id',
           'standard_type', 'pActivity', 'pActivity_min', 'pActivity_max', 'pActivity_spread',
           'n_measurements', 'label', 'confidence_score', 'source', 'chembl_release',
           'retrieved_date']


def collapse(df, slug, release):
    df = df.assign(pActivity=df['standard_value'].map(p_activity))
    agg = (df.groupby('molecule_chembl_id')
             .agg(canonical_smiles=('canonical_smiles', 'first'),
                  molecule_pref_name=('molecule_pref_name', 'first'),
                  standard_type=('standard_type', lambda s: ','.join(sorted(set(s)))),
                  pActivity=('pActivity', 'median'),
                  pActivity_min=('pActivity', 'min'),
                  pActivity_max=('pActivity', 'max'),
                  n_measurements=('pActivity', 'size'),
                  confidence_score=('confidence_score', 'max'))
             .reset_index())
    agg['label'] = agg['pActivity'].map(label)
    agg['pActivity_spread'] = (agg['pActivity_max'] - agg['pActivity_min']).round(2)
    agg['pActivity'] = agg['pActivity'].round(3)
    agg['target_chembl_id'] = TARGETS[slug]['chembl_id']
    agg['source'] = 'ChEMBL API /activity + /assay'
    agg['chembl_release'] = release
    agg['retrieved_date'] = retrieved
    return agg[COLUMNS].sort_values('pActivity', ascending=False)


for slug, s in state.items():
    banner(slug)
    s['agg'] = collapse(s['df'], slug, s['release'])
    s['steps'].append(('unique molecules (median pActivity)', len(s['agg'])))
    display(s['agg'].head())

## Result

In [ ]:
for slug, s in state.items():
    banner(slug)
    counts = s['agg']['label'].value_counts()
    s['counts'] = counts
    print('Labels')
    for lab in ('active', 'dropped', 'inactive'):
        print(f'  {counts.get(lab, 0):>7}  {lab}')
    if counts.get('inactive', 0):
        active, inactive = counts.get('active', 0), counts['inactive']
        print(f'\nactive:inactive = {active / inactive:.1f}:1  -- a classifier that always says '
              f'"active" would score {active / (active + inactive):.0%} accuracy on the binary rows')

display(Markdown('#### All three targets side by side'))
cascade_table = pd.DataFrame({slug: dict(s['steps']) for slug, s in state.items()})
labels_table = pd.DataFrame({slug: s['counts'] for slug, s in state.items()}).reindex(
    ['active', 'dropped', 'inactive']).fillna(0).astype(int)
display(pd.concat([cascade_table, labels_table]))

The active to inactive ratio is 27.6:1 for p38, 3.8:1 for COX-2 and 6.1:1 for 5-LOX.

## Write

In [ ]:
for slug, s in state.items():
    banner(slug)
    t, agg, counts, steps = TARGETS[slug], s['agg'], s['counts'], s['steps']
    if REFRESH:
        out_csv(slug).parent.mkdir(parents=True, exist_ok=True)
        agg.to_csv(out_csv(slug), index=False, encoding='utf-8')
        print('->', out_csv(slug))

        record = (f'\n## {t["name"]} training set ({retrieved})\n\n'
                  f'- Target: `{t["chembl_id"]}` ({t["name"]}), *Homo sapiens*\n'
                  f'- Source: ChEMBL REST API, `/activity` joined to `/assay` on `assay_chembl_id`\n'
                  f'- ChEMBL release: {s["release"]}\n'
                  f'- Retrieved: {retrieved}\n'
                  f"- Criteria: IC50/Ki, nM, relation '=', confidence_score >= {CONFIDENCE_MIN}, "
                  f'no data_validity_comment, not potential_duplicate\n'
                  f'- Records fetched: {steps[0][1]} -> surviving activities: '
                  f'{s["surviving_activities"]} -> unique molecules: {len(agg)}\n'
                  f"- Labels: active {counts.get('active', 0)}, inactive {counts.get('inactive', 0)}, "
                  f"dropped (5-6 band) {counts.get('dropped', 0)}\n"
                  f'- Licence: ChEMBL data is CC BY-SA 3.0\n'
                  f'- Notebook: `src/1_fetch_p38.ipynb`; raw payload '
                  f'`data/raw/{slug}_activities_raw.json`\n')
        PROVENANCE.parent.mkdir(parents=True, exist_ok=True)
        with PROVENANCE.open('a', encoding='utf-8') as fh:
            fh.write(record)
        print('-> provenance appended to', PROVENANCE)
    else:
        existing = pd.read_csv(out_csv(slug))
        print('REFRESH is False, nothing written.')
        print(f'The committed {out_csv(slug).name} has {len(existing):,} molecules '
              f"(retrieved {existing.retrieved_date.iloc[0]}, {existing.chembl_release.iloc[0]}).")
        same = (len(existing) == len(agg)
                and set(existing.molecule_chembl_id) == set(agg.molecule_chembl_id))
        print(f'Recomputed from the saved payload: {len(agg):,} molecules '
              f"-> {'matches' if same else 'DOES NOT MATCH'}")